# Day 1 — Hands-On Lab: PySpark & Spark SQL with ADLS

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Dataset** | customers_010626.csv — 6,666 rows, 8 columns |
| **Storage** | Azure Data Lake Storage Gen2 |
| **Duration** | 60 minutes |
| **Layer** | Bronze (Delta format) |

### Learning Objectives
- Connect Databricks to ADLS Gen2 using a storage access key
- Explore a DataFrame with PySpark (`printSchema`, `show`, `count`)
- Query data with Spark SQL (temp views, YEAR, TRIM, GROUP BY)
- Apply PySpark transformations (`filter`, `withColumn`, `when/otherwise`)
- Write cleaned data to a Bronze Delta table

---
**Instructions:** Run each cell with **Shift + Enter**. Fill in any `# YOUR CODE HERE` blanks before running.

## Setup: Connect to ADLS

Replace the placeholders below with your values:
- `YOUR_STORAGE_ACCOUNT_NAME` → the name you chose when creating your storage account (e.g. `globalmartvirincy`)
- `YOUR_STORAGE_ACCOUNT_KEY` → copied from Azure Portal → your storage account → Security + networking → Access Keys → key1

> ⚠️ **Never share your key or push it to GitHub.** It gives full access to your storage account.

In [ ]:
# ─── ADLS Connection Setup ─────────────────────────────────────────────────
storage_account_name = "YOUR_STORAGE_ACCOUNT_NAME"   # ← replace with your storage account name
container_name       = "amazon-data"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"    # ← replace with your access key

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

base_path    = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"
raw_path     = f"{base_path}/raw/customers"
bronze_path  = f"{base_path}/bronze/customers"

print("Connection successful!")
print(f"Raw path:    {raw_path}")
print(f"Bronze path: {bronze_path}")

---
## Phase A — Understand the Data

**Goal:** Read the customer CSV from ADLS and understand its structure.

### A1 — Read the CSV

Use `spark.read.csv()` to load the customer file. We pass two options:
- `header=true` → first row is column names
- `inferSchema=true` → Spark auto-detects data types

In [ ]:
# Read customers CSV from ADLS
customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{raw_path}/customers_010626.csv")
)

print(f"Total rows: {customers_df.count()}")

### A2 — Explore the Schema

Print the schema (column names + data types) and preview the first 5 rows.

In [ ]:
# Print schema
customers_df.printSchema()

# Show first 5 rows with full values
customers_df.show(5, truncate=False)

**Q: List all 8 column names and their Spark-detected data types below.**

| Column | Data Type |
|--------|-----------|
| CustomerID | |
| FirstName | |
| LastName | |
| Email | |
| PhoneNumber | |
| DateOfBirth | |
| RegistrationDate | |
| PreferredPaymentMethodID | |

---
## Phase B — Spark SQL

**Goal:** Register the DataFrame as a SQL view and query it using standard SQL.

First, register the temp view — this makes the DataFrame queryable as a SQL table.

In [ ]:
# Register as a temporary SQL view
customers_df.createOrReplaceTempView("customers")

print("Temp view 'customers' registered. You can now query it using SQL.")

### B1 — Payment Method Distribution

How many customers prefer each payment method? Sort by count descending.

In [ ]:
result = spark.sql("""
    SELECT PreferredPaymentMethodID,
           COUNT(*) AS customer_count
    FROM customers
    GROUP BY PreferredPaymentMethodID
    ORDER BY customer_count DESC
""")
result.show()

**Q: Which payment method has the highest number of customers? How many?**

*Your answer:* _______________

### B2 — Customers Registered Per Year

**Hint:** Use `YEAR(RegistrationDate)` to extract the year. Sort newest first.

In [ ]:
result = spark.sql("""
    SELECT YEAR(RegistrationDate) AS registration_year,
           COUNT(*) AS customer_count
    FROM customers
    GROUP BY registration_year
    ORDER BY registration_year DESC
""")
result.show(30)

**Q: In which year did the most customers register? How many?**

*Your answer:* _______________

### B3 — Customers Born in the 1990s

Write a Spark SQL query to find customers born between 1 Jan 1990 and 31 Dec 1999.

**Hint:** Use `WHERE DateOfBirth BETWEEN 'YYYY-MM-DD' AND 'YYYY-MM-DD'`

In [ ]:
result = spark.sql("""
    -- YOUR SQL QUERY HERE
    -- Count customers born between 1990-01-01 and 1999-12-31
    SELECT COUNT(*) AS customers_born_1990s
    FROM customers
    WHERE DateOfBirth BETWEEN '1990-01-01' AND '1999-12-31'
""")
result.show()

**Q: How many customers were born in the 1990s?**

*Your answer:* _______________

---
## Phase C — PySpark Transformations

**Goal:** Use PySpark's DataFrame API to filter rows, add computed columns, and fix data quality issues.

### C1 — Filter + Add Age Column

**Task 1:** Filter customers who registered in **2020 or later**.

**Task 2:** Add an `age` column calculated from `DateOfBirth` to today.

In [ ]:
from pyspark.sql.functions import col, year, floor, datediff, current_date, when, trim, length
from pyspark.sql.functions import min as spark_min, max as spark_max, avg

# Task 1: Filter customers who registered 2020 or later
recent_customers = customers_df.filter(year(col("RegistrationDate")) >= 2020)
print(f"Customers registered 2020 or later: {recent_customers.count()}")

In [ ]:
# Task 2: Add 'age' column
customers_with_age = customers_df.withColumn(
    "age",
    floor(datediff(current_date(), col("DateOfBirth")) / 365)
)

customers_with_age.agg(
    spark_min("age").alias("youngest"),
    spark_max("age").alias("oldest"),
    avg("age").alias("average_age")
).show()

**Q:**
1. How many customers registered in 2020 or later? _______________
2. Age of the youngest customer? ___ / oldest? ___

### C2 — Classify Customers into Loyalty Tiers

Add a `LoyaltyTier` column:
- Registered **before 2010** → `"Gold"`
- Registered **2010–2019** → `"Silver"`
- Registered **2020 or later** → `"Bronze"`

Use `when / otherwise`.

In [ ]:
customers_tiered = customers_with_age.withColumn(
    "LoyaltyTier",
    when(year(col("RegistrationDate")) < 2010, "Gold")
    .when(year(col("RegistrationDate")) < 2020, "Silver")
    .otherwise("Bronze")
)

customers_tiered.groupBy("LoyaltyTier").count().orderBy("LoyaltyTier").show()

**Q: How many customers fall into each tier?**

| Tier | Count |
|------|-------|
| Bronze | |
| Gold | |
| Silver | |

### C3 — Fix Data Quality: Trim Spaces

The `FirstName` column has trailing spaces in some rows (e.g. `"Ahana "` instead of `"Ahana"`). Fix this using `trim()`.

In [ ]:
# Find how many rows have the issue
dirty = customers_tiered.filter(
    length(col("FirstName")) != length(trim(col("FirstName")))
)
print(f"Rows with leading/trailing spaces in FirstName: {dirty.count()}")

# Fix: trim FirstName and Email
customers_clean = customers_tiered.withColumn(
    "FirstName", trim(col("FirstName"))
).withColumn(
    "Email", trim(col("Email"))
)

print("\nCleaned sample:")
customers_clean.select("CustomerID", "FirstName", "Email").show(5, truncate=False)

**Q:**
1. How many rows had spaces in `FirstName`? _______________
2. Why is trimming important before writing to the Silver layer?

*Your answer:* _______________

---
## Phase D — Write to Bronze

**Goal:** Persist the cleaned DataFrame as a Delta table in the Bronze layer.

### D1 — Write to Bronze as Delta

In [ ]:
# Write cleaned data to Bronze — Delta format
customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

print(f"Written to: {bronze_path}")

### D2 — Verify the Bronze Table

In [ ]:
# Read back from Bronze and verify row count
bronze_df = spark.read.format("delta").load(bronze_path)

print(f"Bronze row count: {bronze_df.count()}")
bronze_df.show(3, truncate=False)

**Q: Does the Bronze row count match the source CSV? What is the count?**

*Your answer:* _______________

---
## Submission Checklist

Before uploading this notebook, fill in each blank and verify the checklist.

> ⚠️ **Replace your real `storage_account_key` value with the placeholder `YOUR_STORAGE_ACCOUNT_KEY` in the setup cell before uploading.**

```
Submission Checklist
────────────────────────────────────────────────────────
✅ Storage account created with Hierarchical Namespace enabled
✅ Container 'amazon-data' with raw/customers/ folder structure
✅ customers_010626.csv uploaded to ADLS
✅ Databricks connected — 'Connection successful!' confirmed
── Total row count:                 ______
── Top payment method:              ______
── Year with most registrations:    ______
── Bronze tier count:               ______
── Silver tier count:               ______
── Gold tier count:                 ______
── FirstName rows with spaces:      ______
── Bronze Delta row count:          ______
────────────────────────────────────────────────────────
```